In [52]:
from google.colab import userdata
from google.colab import drive
import urllib.request
import pandas as pd
import os

# Configurações do GIT
token = userdata.get('GITHUB_TOKEN')
usuario = "muriloklein"
repo = "tcc"

if not os.path.exists("/content/tcc"):
    !git clone https://{token}@github.com/{usuario}/{repo}.git
%cd /content/tcc

/content/tcc


In [53]:
# Montar o Google Drive
drive.mount('/content/drive')

# Cria a pasta onde os dados brutos vão ficar (se ainda não existir)
DADOS_DIR = "/content/drive/MyDrive/TCC/dados/raw"
os.makedirs(DADOS_DIR, exist_ok=True)

print(f"Pasta de dados pronta em: {DADOS_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pasta de dados pronta em: /content/drive/MyDrive/TCC/dados/raw


In [54]:
# Baixar os arquivos conn.log.labeled dos cenários escolhidos
cenarios = {
    "capture_34_1_DDoS_PortScan": (
        "https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/"
        "IndividualScenarios/CTU-IoT-Malware-Capture-34-1/bro/conn.log.labeled"
    ),
    "capture_9_1_PortScan": (
        "https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/"
        "IndividualScenarios/CTU-IoT-Malware-Capture-9-1/bro/conn.log.labeled"
    ),
    "capture_36_1_Okiru": (
        "https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/"
        "IndividualScenarios/CTU-IoT-Malware-Capture-36-1/bro/conn.log.labeled"
    ),
    "capture_4_1_Benign_Honeypot": (
        "https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/"
        "IndividualScenarios/CTU-Honeypot-Capture-4-1/bro/conn.log.labeled"
    ),
    "capture_5_1_Benign_Honeypot": (
      "https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/"
      "IndividualScenarios/CTU-Honeypot-Capture-5-1/bro/conn.log.labeled"
    ),
    "capture_7_1_Benign_Honeypot": (
      "https://mcfp.felk.cvut.cz/publicDatasets/IoT-23-Dataset/"
      "IndividualScenarios/CTU-Honeypot-Capture-7-1/Somfy-01/bro/conn.log.labeled"
    ),
}

for nome, url in cenarios.items():
    destino = os.path.join(DADOS_DIR, f"{nome}.log.labeled")

    # Se baixado antes, não baixa de novo
    if os.path.exists(destino):
        print(f"[OK] {nome} já existe, pulando download.")
        continue

    print(f"Baixando {nome}...")
    try:
        urllib.request.urlretrieve(url, destino)
        print(f"[OK] {nome} salvo em {destino}")
    except Exception as e:
        print(f"[ERRO] Falha ao baixar {nome}: {e}")


[OK] capture_34_1_DDoS_PortScan já existe, pulando download.
[OK] capture_9_1_PortScan já existe, pulando download.
[OK] capture_36_1_Okiru já existe, pulando download.
[OK] capture_4_1_Benign_Honeypot já existe, pulando download.
[OK] capture_5_1_Benign_Honeypot já existe, pulando download.
[OK] capture_7_1_Benign_Honeypot já existe, pulando download.


In [55]:
# Entender o cabeçalho do arquivo antes de carregar
caminho_exemplo = os.path.join(DADOS_DIR, "capture_34_1_DDoS_PortScan.log.labeled")

with open(caminho_exemplo, "r") as f:
    for i, linha in enumerate(f):
        print(f"{i}: {linha.strip()}")
        if i > 10:  # só as 10 primeiras linhas, o suficiente pra ver o cabeçalho
            break

# "#fields" -> lista os nomes das colunas
# "#separator \x09" -> confirma que é tab

0: #separator \x09
1: #set_separator	,
2: #empty_field	(empty)
3: #unset_field	-
4: #path	conn
5: #open	2019-03-15-14-50-49
6: #fields	ts	uid	id.orig_h	id.orig_p	id.resp_h	id.resp_p	proto	service	duration	orig_bytes	resp_bytes	conn_state	local_orig	local_resp	missed_bytes	history	orig_pkts	orig_ip_bytes	resp_pkts	resp_ip_bytes	tunnel_parents	label	det_label
7: #types	time	string	addr	port	addr	port	enum	string	interval	count	count	string	bool	bool	count	string	count	count	count	count	set[string]	string	string
8: 1545403816.962094	CrDn63WjJEmrWGjqf	192.168.1.195	41040	185.244.25.235	80	tcp	-	3.139211	0	0	S0	-	-	0	S	3	180	0	0	-	Benign	-
9: 1545403824.181240	CY9lJW3gh1Eje4usP6	192.168.1.195	41040	185.244.25.235	80	tcp	-	-	-	-	S0	-	-	0	S	1	60	0	0	-	Benign	-
10: 1545403832.341331	CcFXLynukEDnUlvgl	192.168.1.195	41040	185.244.25.235	80	tcp	-	-	-	-	S0	-	-	0	S	1	60	0	0	-	Benign	-
11: 1545403848.981338	CDrkrSobGYxHhYfth	192.168.1.195	41040	185.244.25.235	80	tcp	http	1.477656	149	128252	SF	-	-	2

In [56]:
# Carregar o arquivo corretamente com pandas (VERSÃO OTIMIZADA em memória)
import gc

colunas_usadas = [
    "id.orig_p", "id.resp_p", "proto", "service", "duration",
    "orig_bytes", "resp_bytes", "conn_state", "local_orig", "local_resp",
    "missed_bytes", "history", "orig_pkts", "orig_ip_bytes",
    "resp_pkts", "resp_ip_bytes", "label", "detailed-label"
]

# Nomes de coluna padrão do conn.log do Zeek + as 2 colunas extras do IoT-23.
colunas_completas = [
    "ts", "uid", "id.orig_h", "id.orig_p", "id.resp_h", "id.resp_p",
    "proto", "service", "duration", "orig_bytes", "resp_bytes",
    "conn_state", "local_orig", "local_resp", "missed_bytes",
    "history", "orig_pkts", "orig_ip_bytes", "resp_pkts", "resp_ip_bytes",
    "tunnel_parents", "label", "detailed-label"
]

caminho_34 = os.path.join(DADOS_DIR, "capture_34_1_DDoS_PortScan.log.labeled")
caminho_36 = os.path.join(DADOS_DIR, "capture_36_1_Okiru.log.labeled")
caminho_5_1 = os.path.join(DADOS_DIR, "capture_5_1_Benign_Honeypot.log.labeled")
caminho_4_1 = os.path.join(DADOS_DIR, "capture_4_1_Benign_Honeypot.log.labeled")
caminho_9_1 = os.path.join(DADOS_DIR, "capture_9_1_PortScan.log.labeled")
caminho_7_1 = os.path.join(DADOS_DIR, "capture_7_1_Benign_Honeypot.log.labeled")

# dtype otimizado: colunas repetitivas viram 'category' (muito mais leve que string/object)
# e colunas numéricas usam tipos menores (int32 em vez de int64/float64)
dtype_otimizado = {
    "proto": "category",
    "service": "category",
    "conn_state": "category",
    "local_orig": "category",
    "local_resp": "category",
    "history": "category",
    "label": "category",
    "detailed-label": "category",
    "id.orig_p": "int32",
    "id.resp_p": "int32",
    "missed_bytes": "int32",
    "orig_pkts": "int32",
    "orig_ip_bytes": "int32",
    "resp_pkts": "int32",
    "resp_ip_bytes": "int32",
}

config = {
    "sep": r"\s+", # colunas separadas por espaço/tab (o header usa \x09; Mas as últimas colunas às vezes vêm separadas por espaço simples - \s+ lida com os dois casos)
    "comment": "#", # ignora todas as linhas de cabeçalho que começam com #
    "header": None, # não tem cabeçalho "normal" de CSV, vamos nomear na mão
    "names": colunas_completas, # nomes de todas as colunas do arquivo
    "usecols": colunas_usadas, # mas só carrega estas na memória
    "na_values": "-", # o Zeek usa "-" para representar valor ausente
    "dtype": dtype_otimizado, # <-- reduz drasticamente o uso de RAM
}


In [57]:
# Função utilitária: carrega um cenário já otimizado e imprime um resumo
def carregar_cenario(caminho, nome_origem, filtro_excluir_detailed_label=None):
    df = pd.read_csv(caminho, **config)

    # 'label' e 'detailed-label' já vêm como category; padroniza para minúsculo
    df["label"] = df["label"].astype("string").str.lower().astype("category")
    df["detailed-label"] = df["detailed-label"].astype("string").str.lower().astype("category")

    if filtro_excluir_detailed_label is not None:
        df = df[df["detailed-label"] != filtro_excluir_detailed_label]

    df["origem_cenario"] = nome_origem

    print(f"Formato do DataFrame ({nome_origem}): {df.shape}")
    print(f"Uso de memória: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(df["label"].value_counts())
    print("-" * 60)

    return df


In [58]:
df_34 = carregar_cenario(caminho_34, "capture_34_1")


Formato do DataFrame (capture_34_1): (23145, 19)
Uso de memória: 2.67 MB
label
malicious    21222
benign        1923
Name: count, dtype: int64
------------------------------------------------------------


In [59]:
df_36 = carregar_cenario(caminho_36, "capture_36_1", filtro_excluir_detailed_label="okiru-attack")


Formato do DataFrame (capture_36_1): (13645095, 19)
Uso de memória: 1678.68 MB
label
malicious    13642432
benign           2663
Name: count, dtype: int64
------------------------------------------------------------


In [60]:
df_5_1 = carregar_cenario(caminho_5_1, "capture_5_1")


Formato do DataFrame (capture_5_1): (1374, 19)
Uso de memória: 0.16 MB
label
benign    1374
Name: count, dtype: int64
------------------------------------------------------------


In [61]:
df_4_1 = carregar_cenario(caminho_4_1, "capture_4_1")


Formato do DataFrame (capture_4_1): (452, 19)
Uso de memória: 0.05 MB
label
benign    452
Name: count, dtype: int64
------------------------------------------------------------


In [62]:
df_9_1 = carregar_cenario(caminho_9_1, "capture_9_1")


Formato do DataFrame (capture_9_1): (6378293, 19)
Uso de memória: 729.94 MB
label
malicious    6355745
benign         22548
Name: count, dtype: int64
------------------------------------------------------------


In [63]:
df_7_1 = carregar_cenario(caminho_7_1, "capture_7_1")

Formato do DataFrame (capture_7_1): (130, 19)
Uso de memória: 0.02 MB
label
benign    130
Name: count, dtype: int64
------------------------------------------------------------


In [64]:
# Une todos em um único DataFrame, empilhando as linhas
# (a coluna 'origem_cenario' já foi adicionada dentro de carregar_cenario)
df_completo = pd.concat(
    [df_34, df_36, df_9_1, df_4_1, df_5_1, df_7_1],
    ignore_index=True   # gera um novo índice sequencial, em vez de repetir os índices originais
)

# Libera a memória dos DataFrames individuais, já que agora estão duplicados dentro de df_completo
del df_34, df_36, df_9_1, df_4_1, df_5_1, df_7_1
gc.collect()

print(f"Formato final: {df_completo.shape}")
print(f"Uso de memória total: {df_completo.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(df_completo["origem_cenario"].value_counts())
print(df_completo["label"].value_counts())


Formato final: (20048489, 19)
Uso de memória total: 7992.63 MB
origem_cenario
capture_36_1    13645095
capture_9_1      6378293
capture_34_1       23145
capture_5_1         1374
capture_4_1          452
capture_7_1          130
Name: count, dtype: int64
label
malicious    20019399
benign          29090
Name: count, dtype: Int64


In [65]:
# Cria uma coluna auxiliar só para a amostragem,
# e para linhas benignas (que têm detailed-label vazio) usa "benign" como rótulo do grupo
df_completo["classe_amostra"] = (
    df_completo["detailed-label"]
    .astype("string")
    .fillna("benign")
)

print(df_completo["classe_amostra"].value_counts())

classe_amostra
okiru                        13626744
partofahorizontalportscan     6355867
benign                          29090
c&c-heartbeat                   15688
ddos                            14394
c&c                              6706
Name: count, dtype: Int64


In [66]:
CAP_POR_CLASSE = 20000
RANDOM_STATE = 42
CLASSES_SEM_TETO = {"benign"}   # não corta o benigno, mesmo acima do teto

grupos_amostrados = []
for classe, grupo in df_completo.groupby("classe_amostra", observed=True):
    if classe not in CLASSES_SEM_TETO and len(grupo) > CAP_POR_CLASSE:
        grupo = grupo.sample(n=CAP_POR_CLASSE, random_state=RANDOM_STATE)
    grupos_amostrados.append(grupo)

df_amostrado = pd.concat(grupos_amostrados, ignore_index=True)
df_amostrado = df_amostrado.drop(columns=["classe_amostra"])

del df_completo
import gc; gc.collect()

print(f"Formato após amostragem: {df_amostrado.shape}")
print(df_amostrado["detailed-label"].value_counts(dropna=False))
print(f"Uso de memória: {df_amostrado.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

Formato após amostragem: (105878, 19)
detailed-label
<NA>                         29090
partofahorizontalportscan    20000
okiru                        20000
c&c-heartbeat                15688
ddos                         14394
c&c                           6706
Name: count, dtype: Int64
Uso de memória: 42.09 MB


In [67]:
# Checkpoint para salvar os dados

PROCESSED_DIR = "/content/drive/MyDrive/TCC/dados/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

caminho_checkpoint = os.path.join(PROCESSED_DIR, "dataset_amostrado_v1.parquet")
df_amostrado.to_parquet(caminho_checkpoint, index=False)

print(f"Checkpoint salvo em: {caminho_checkpoint}")
print(f"Tamanho do arquivo: {os.path.getsize(caminho_checkpoint) / 1024**2:.2f} MB")

Checkpoint salvo em: /content/drive/MyDrive/TCC/dados/processed/dataset_amostrado_v1.parquet
Tamanho do arquivo: 0.72 MB


In [68]:
# import pandas as pd
# import os

caminho_checkpoint = os.path.join(PROCESSED_DIR, "dataset_amostrado_v1.parquet")
df_amostrado = pd.read_parquet(caminho_checkpoint)

print(f"Formato recarregado: {df_amostrado.shape}")
print(df_amostrado["detailed-label"].value_counts(dropna=False))

Formato recarregado: (105878, 19)
detailed-label
<NA>                         29090
partofahorizontalportscan    20000
okiru                        20000
c&c-heartbeat                15688
ddos                         14394
c&c                           6706
Name: count, dtype: Int64


In [69]:
# Tratamento de valores ausentes

# Descarta colunas 100% vazias
df_amostrado = df_amostrado.drop(columns=["local_orig", "local_resp"])

# Cria indicadores antes de preencher
for col in ["duration", "orig_bytes", "resp_bytes"]:
    df_amostrado[f"{col}_ausente"] = df_amostrado[col].isna()

# Preenche os numéricos ausentes com 0
df_amostrado[["duration", "orig_bytes", "resp_bytes"]] = (
    df_amostrado[["duration", "orig_bytes", "resp_bytes"]].fillna(0)
)

# Preenche categóricas ausentes com rótulo explícito
for col in ["service", "history"]:
    if df_amostrado[col].dtype.name == "category":
        df_amostrado[col] = df_amostrado[col].cat.add_categories(["desconhecido"])
    df_amostrado[col] = df_amostrado[col].fillna("desconhecido")

# Confirma que só restou NaN em detailed-label (esperado, são as linhas benignas)
print(df_amostrado.isna().sum())

id.orig_p                 0
id.resp_p                 0
proto                     0
service                   0
duration                  0
orig_bytes                0
resp_bytes                0
conn_state                0
missed_bytes              0
history                   0
orig_pkts                 0
orig_ip_bytes             0
resp_pkts                 0
resp_ip_bytes             0
label                     0
detailed-label        29090
origem_cenario            0
duration_ausente          0
orig_bytes_ausente        0
resp_bytes_ausente        0
dtype: int64


In [70]:
# Outro checkpoint
caminho_checkpoint_limpo = os.path.join(PROCESSED_DIR, "dataset_limpo_v1.parquet")
df_amostrado.to_parquet(caminho_checkpoint_limpo, index=False)
print(f"Checkpoint salvo em: {caminho_checkpoint_limpo}")

Checkpoint salvo em: /content/drive/MyDrive/TCC/dados/processed/dataset_limpo_v1.parquet


In [71]:
colunas_categoricas = ["proto", "service", "conn_state", "history", "origem_cenario"]

for col in colunas_categoricas:
    print(f"{col}: {df_amostrado[col].nunique()} valores distintos")
    print(df_amostrado[col].value_counts().head(10))  # os 10 mais comuns
    print("-" * 60)

proto: 3 valores distintos
proto
tcp     82065
udp     20647
icmp     3166
Name: count, dtype: int64
------------------------------------------------------------
service: 6 valores distintos
service
desconhecido    100824
dns               2871
irc               1641
http               278
dhcp               177
ssl                 87
Name: count, dtype: int64
------------------------------------------------------------
conn_state: 13 valores distintos
conn_state
S0      78030
OTH     17446
SF       4216
REJ      4068
S3       1611
RSTO      205
RSTR      135
SH        106
S1         19
SHR        14
Name: count, dtype: int64
------------------------------------------------------------
history: 109 valores distintos
history
S               60983
D               17040
C               14252
Sr               4068
Dd               3597
desconhecido     3166
ShAdDaf          1477
ShR               172
ShAFf             172
ShAdDafF          143
Name: count, dtype: int64
--------------------

In [72]:
# Agrupando categorias
# Define o corte: manter categorias com pelo menos 500 ocorrências, por exemplo
LIMIAR_MINIMO = 500

contagem = df_amostrado["history"].value_counts()
categorias_frequentes = contagem[contagem >= LIMIAR_MINIMO].index.tolist()

print(f"Categorias mantidas ({len(categorias_frequentes)}): {categorias_frequentes}")
print(f"Categorias agrupadas em 'outros': {df_amostrado['history'].nunique() - len(categorias_frequentes)}")

# Aplica: quem não está na lista de frequentes vira "outros"
df_amostrado["history"] = df_amostrado["history"].astype("string")
df_amostrado["history"] = df_amostrado["history"].where(
    df_amostrado["history"].isin(categorias_frequentes),
    "outros"
)

print(df_amostrado["history"].value_counts())

Categorias mantidas (7): ['S', 'D', 'C', 'Sr', 'Dd', 'desconhecido', 'ShAdDaf']
Categorias agrupadas em 'outros': 102
history
S               60983
D               17040
C               14252
Sr               4068
Dd               3597
desconhecido     3166
ShAdDaf          1477
outros           1295
Name: count, dtype: Int64


In [73]:
# One hot encoding
colunas_para_onehot = ["proto", "service", "conn_state", "history"]

#label e detailed-label não devem passar por one-hot — eles são os alvos que o modelo vai tentar prever (y), não características de entrada (X). Da mesma forma, origem_cenario continua fora (é metadado de rastreio, não feature, como combinamos). Por isso eu não coloquei nenhum dos três na lista colunas_para_onehot — repara e confirma isso antes de rodar.

df_encoded = pd.get_dummies(
    df_amostrado,
    columns=colunas_para_onehot,
    prefix=colunas_para_onehot, # prefixo nas novas colunas, ex: proto_tcp, service_dns
    dtype="int8" # 0/1 como número pequeno, economiza memória
)

print(f"Formato antes: {df_amostrado.shape}")
print(f"Formato depois: {df_encoded.shape}")
print(df_encoded.columns.tolist())

Formato antes: (105878, 20)
Formato depois: (105878, 46)
['id.orig_p', 'id.resp_p', 'duration', 'orig_bytes', 'resp_bytes', 'missed_bytes', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'label', 'detailed-label', 'origem_cenario', 'duration_ausente', 'orig_bytes_ausente', 'resp_bytes_ausente', 'proto_icmp', 'proto_tcp', 'proto_udp', 'service_desconhecido', 'service_dhcp', 'service_dns', 'service_http', 'service_irc', 'service_ssl', 'conn_state_OTH', 'conn_state_REJ', 'conn_state_RSTO', 'conn_state_RSTOS0', 'conn_state_RSTR', 'conn_state_RSTRH', 'conn_state_S0', 'conn_state_S1', 'conn_state_S2', 'conn_state_S3', 'conn_state_SF', 'conn_state_SH', 'conn_state_SHR', 'history_C', 'history_D', 'history_Dd', 'history_S', 'history_ShAdDaf', 'history_Sr', 'history_desconhecido', 'history_outros']


In [74]:
# Checkpoint

caminho_checkpoint_encoded = os.path.join(PROCESSED_DIR, "dataset_encoded_v1.parquet")
df_encoded.to_parquet(caminho_checkpoint_encoded, index=False)
print(f"Checkpoint salvo em: {caminho_checkpoint_encoded}")

Checkpoint salvo em: /content/drive/MyDrive/TCC/dados/processed/dataset_encoded_v1.parquet


In [75]:
!pwd

!cd tcc

!pwd

/content/tcc
/bin/bash: line 1: cd: tcc: No such file or directory
/content/tcc
